In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("student_performance_data.csv")
def load_dataset(file_path):
    """
    Load a dataset and display basic information about it.
    
    Parameters:
    file_path (str): Path to the dataset file.
    
    Returns:
    DataFrame: The loaded dataset.
    """
    try:
        # Try to infer the file type from extension
        if file_path.endswith('.csv'):
            df = pd.read_csv(file_path)
        elif file_path.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(file_path)
        else:
            print(f"Unsupported file type: {file_path}")
            return None
        
        print("\nDataset Shape:", df.shape)
        print("\nFirst 5 Rows:")
        print(df.head())
        print("\nData Types:")
        print(df.dtypes)
        print("\nSummary Statistics:")
        print(df.describe())
        print("\nMissing Values per Column:")
        print(df.isnull().sum())
        return df
    
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None
    




In [6]:
def check_data_quality(df):
    """
    Check data quality issues such as missing values, duplicates, and outliers.
    
    Parameters:
    df (DataFrame): The dataset to check.
    
    Returns:
    dict: A dictionary containing data quality metrics.
    """
    quality_report = {}
    
    # Check missing values
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    quality_report['missing_values'] = {
        'count': missing_values,
        'percentage': missing_percentage
    }
    
    # TODO: Check duplicates
    # 1. Count the number of duplicate rows
    # 2. Calculate the percentage of duplicate rows
    # 3. Add this information to the quality_report dictionary
    
    # TODO: Check for potential outliers in numeric columns
    # 1. For each numeric column, calculate Q1, Q3, and IQR
    # 2. Define lower and upper bounds (Q1 - 1.5*IQR and Q3 + 1.5*IQR)
    # 3. Count values outside these bounds
    # 4. Add this information to the quality_report dictionary
    
    return quality_report


In [ ]:
def check_data_quality(df):
    """
    Check data quality issues such as missing values, duplicates, and outliers.
    
    Parameters:
    df (DataFrame): The dataset to check.
    
    Returns:
    dict: A dictionary containing data quality metrics.
    """
    quality_report = {}
    
    # Check missing values
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    quality_report['missing_values'] = {
        'count': missing_values,
        'percentage': missing_percentage
    }
    
    # TODO: Check duplicates
    # 1. Count the number of duplicate rows
    # 2. Calculate the percentage of duplicate rows
    # 3. Add this information to the quality_report dictionary
    
    # TODO: Check for potential outliers in numeric columns
    # 1. For each numeric column, calculate Q1, Q3, and IQR
    # 2. Define lower and upper bounds (Q1 - 1.5*IQR and Q3 + 1.5*IQR)
    # 3. Count values outside these bounds
    # 4. Add this information to the quality_report dictionary
    outliers = {}
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
        outliers[col] = {
            'count': outliers_count,
            'percentage': (outliers_count / len(df)) * 100,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
    quality_report['potential_outliers'] = outliers
    return quality_report


In [ ]:
def clean_dataset(df, quality_report=None):
    """
    Clean the dataset based on identified issues.
    
    Parameters:
    df (DataFrame): The dataset to clean.
    quality_report (dict, optional): Data quality report generated by check_data_quality.
    
    Returns:
    DataFrame: The cleaned dataset.
    """
    # Create a copy to avoid modifying the original
    cleaned_df = df.copy()
    
    # Generate quality report if not provided
    if quality_report is None:
        quality_report = check_data_quality(df)
    
    print("Starting data cleaning process...")
    
    # TODO: Handle duplicates
    # 1. Remove duplicate rows
    # 2. Print the number of rows removed

    initial_rows = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates()
    rows_removed = initial_rows - len(cleaned_df)
    print(f"Removed {rows_removed} duplicate rows.")
    
    
    # TODO: Handle missing values
    # 1. For each column with missing values:
    #    a. If missing percentage is high (e.g., > 50%), consider recommending to drop the column
    #    b. For numeric columns, impute with median
    #    c. For categorical columns, impute with mode
    # 2. Print the imputation details for each column
    for col in cleaned_df.columns:
        missing_pct = quality_report['missing_values']['percentage'][col]
        
        # If missing percentage is high (e.g., > 50%), consider dropping the column
        if missing_pct > 50:
            print(f"Column '{col}' has {missing_pct:.2f}% missing values. Consider dropping this column.")
        
        # For numeric columns, impute with median
        elif col in cleaned_df.select_dtypes(include=['int64', 'float64']).columns:
            if missing_pct > 0:
                median_val = cleaned_df[col].median()
                cleaned_df[col].fillna(median_val, inplace=True)
                print(f"Imputed {missing_pct:.2f}% missing values in '{col}' with median: {median_val}")
        
        # For categorical columns, impute with mode
        elif col in cleaned_df.select_dtypes(include=['object', 'category']).columns:
            if missing_pct > 0:
                mode_val = cleaned_df[col].mode()[0]
                cleaned_df[col].fillna(mode_val, inplace=True)
                print(f"Imputed {missing_pct:.2f}% missing values in '{col}' with mode: {mode_val}")
        
    # TODO: Handle outliers
    # 1. Create flags for outliers in each numeric column
    # 2. Add these flags as new columns in the dataframe
    # 3. Print the number of outliers identified

    outlier_flags = {}
    for col, outlier_info in quality_report['potential_outliers'].items():
        if outlier_info['percentage'] > 0:
            lower_bound = outlier_info['lower_bound']
            upper_bound = outlier_info['upper_bound']
            outlier_flags[f'{col}_outlier'] = (
                (cleaned_df[col] < lower_bound) | (cleaned_df[col] > upper_bound)
            )
            print(f"Flagged {outlier_info['count']} potential outliers in '{col}'")
    
    # Add outlier flags to the dataframe
    if outlier_flags:
        for col, flag in outlier_flags.items():
            cleaned_df[col] = flag
    
    # TODO: Convert data types
    # 1. Try to convert object columns to datetime if they look like dates
    # 2. Try to convert string numeric columns to float
    # 3. Print the successful conversions
    
    for col in cleaned_df.columns:
        # Try to convert object columns to datetime if they look like dates
        if cleaned_df[col].dtype == 'object':
            try:
                # Check if this might be a date column
                if cleaned_df[col].str.contains('-|/|:').any():
                    cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='ignore')
                    if cleaned_df[col].dtype.kind == 'M':  # If conversion was successful
                        print(f"Converted '{col}' to datetime.")
            except:
                pass
        
        # Try to convert string numeric columns to float
        if cleaned_df[col].dtype == 'object':
            try:
                # Remove currency symbols and commas if present
                temp_series = cleaned_df[col].str.replace('[$,]', '', regex=True)
                # Check if all non-NA values can be converted to float
                if pd.to_numeric(temp_series, errors='coerce').notna().all():
                    cleaned_df[col] = pd.to_numeric(temp_series)
                    print(f"Converted '{col}' to numeric.")
            except:
                pass

    print("Data cleaning completed!")
    
    return cleaned_df


In [9]:
def save_cleaned_dataset(df, output_path):
    """
    Save the cleaned dataset to a file.
    
    Parameters:
    df (DataFrame): The dataset to save.
    output_path (str): Path to save the cleaned dataset.
    
    Returns:
    bool: True if saved successfully, False otherwise.
    """
    try:
        # TODO: Save the DataFrame based on file extension
        # 1. If output_path ends with .csv, save as CSV
        # 2. If output_path ends with .xls or .xlsx, save as Excel
        # 3. If output_path doesn't have a recognized extension, save as CSV with .csv appended
        
        print(f"Cleaned dataset saved to: {output_path}")
        return True
    
    except Exception as e:
        print(f"Error saving cleaned dataset: {e}")
        return False

In [10]:
def save_cleaned_dataset(df, output_path):
    """
    Save the cleaned dataset to a file.
    
    Parameters:
    df (DataFrame): The dataset to save.
    output_path (str): Path to save the cleaned dataset.
    
    Returns:
    bool: True if saved successfully, False otherwise.
    """
    try:
        # TODO: Save the DataFrame based on file extension
        # 1. If output_path ends with .csv, save as CSV
        # 2. If output_path ends with .xls or .xlsx, save as Excel
        # 3. If output_path doesn't have a recognized extension, save as CSV with .csv appended
        
        print(f"Cleaned dataset saved to: {output_path}")
        return True
    
    except Exception as e:
        print(f"Error saving cleaned dataset: {e}")
        return False